# notebook 16 — (3+1) 最小模型 R×T³ の幾何構成

nb15 で追加原理探索が一巡し、目標 (3+1) は「**被覆 II（時間・非コンパクト、確立）＋ 次元原理
（空間3、未確立）**」の2本柱に整理された。次元4の原理化（nb15 open-1）は未解決のままだが、
本notebookはそれを**仮に認めた上で**（独立設定本数 p=4 を入力として受け入れ）、被覆 II の時間 R と
空間 T³ を合わせた **R×T³** が (3+1) 時空として整合的に振る舞うかを構成して検証する
（nb15 open-2）。

**位置づけ（条件つき成果）**：これは「なぜ4か」には答えない。「4 を認めれば、その幾何は
(3+1) 時空として無矛盾に組める」という条件つきの確認である。本丸 no-go が「与件の枠内では」
という条件つき確定だったのと同じ性格の成果。構成が整合的だと示せれば、次元原理（open-1）が
目指すべき**到達目標 R×T³ を具体的に固定**できる。

**構成方針**（既確立部品の組み合わせ）：
- 空間 = T³：独立3設定（nb15 立場B の SO(3) 3軸）の角度 (φ1,φ2,φ3)、各 S¹ は周期的・コンパクト。
- 時間 = R：被覆座標 t（nb13 II）、全順序・非有界・非コンパクト。
- 距離は各因子で加法分解（nb04 A-4）。

**規律（教訓D）**：時間は『埋め込みで復元』せず『被覆の階数として与える』（nb13 13.2）。
混合 squared interval を一括 MDS にかけると負定値埋め込みが時間を多次元に割る人工物が出る
（本notebook 16.1 で実演）。時間と空間を別々に構成するのが正しい（D-3、作った量の素性確認）。

## 16.0 セットアップ

In [1]:
import numpy as np
from numpy.linalg import eigh
np.set_printoptions(precision=4, suppress=True, linewidth=120)

N = 3  # 与件（QCD との同定）
print("setup done. N =", N)

setup done. N = 3


## 16.1 素朴な構成の罠：混合 squared interval は時間を多次元に割る

まず**誤った**構成を実演して罠を明示する。ミンコフスキー的 squared interval
`ds² = −dt² + Σ d_space²` を作り一括でローレンツ埋め込みすると、時間が単一にならないことを見る。
これは nb13 13.2 で確認した負定値埋め込みの人工物で、避けるべき構成である。

In [2]:
nt, nph = 6, 5
t = np.linspace(0, 4*np.pi, nt)
ph = np.linspace(0, 2*np.pi, nph, endpoint=False)
T, P1, P2, P3 = np.meshgrid(t, ph, ph, ph, indexing='ij')
pts = np.stack([T.ravel(), P1.ravel(), P2.ravel(), P3.ravel()], 1)
M = pts.shape[0]

def axis_d2(col, periodic=True):
    dx = pts[:, col][:, None] - pts[:, col][None, :]
    a = np.abs(np.cos(2*dx))/(2*N)
    with np.errstate(divide='ignore'):
        d = -np.log(a)
    d[~np.isfinite(d)] = 0.0; np.fill_diagonal(d, 0.0)
    return d**2

dsp2 = axis_d2(1) + axis_d2(2) + axis_d2(3)
dt = pts[:, 0][:, None] - pts[:, 0][None, :]
D2 = -(dt**2) + dsp2; np.fill_diagonal(D2, 0.0)

J = np.eye(M) - np.ones((M, M))/M
B = -0.5 * J @ D2 @ J; B = 0.5*(B + B.T)
ev = np.sort(eigh(B)[0])
print(f"混合一括埋め込み（誤った構成）の符号数: "
      f"spacelike={int((ev>1e-6).sum())}, timelike={int((ev<-1e-6).sum())}")
print(f"  → timelike が複数。単一時間にならない（nb13 13.2 の人工物の再現）。")
print(f"  時間を埋め込みで復元しようとするのが誤り。次セルで正しく構成する。")

混合一括埋め込み（誤った構成）の符号数: spacelike=742, timelike=7
  → timelike が複数。単一時間にならない（nb13 13.2 の人工物の再現）。
  時間を埋め込みで復元しようとするのが誤り。次セルで正しく構成する。


## 16.2 正しい構成：時間と空間を別々に

時間は被覆座標 t（スカラー）を直接1次元の時間として与える（timelike=1 が構成的に保証）。
空間は T³ の `|C|` を MDS で復元する。これが nb13 13.2 の正しい手続きである。

In [3]:
nph = 6
ph = np.linspace(0, 2*np.pi, nph, endpoint=False)
P1, P2, P3 = np.meshgrid(ph, ph, ph, indexing='ij')
sp = np.stack([P1.ravel(), P2.ravel(), P3.ravel()], 1)
Ms = sp.shape[0]

def sp_axis_d2(col):
    dphi = sp[:, col][:, None] - sp[:, col][None, :]
    a = np.abs(np.cos(2*dphi))/(2*N)
    with np.errstate(divide='ignore'):
        d = -np.log(a)
    d[~np.isfinite(d)] = 0.0; np.fill_diagonal(d, 0.0)
    return d**2

dsp2 = sp_axis_d2(0) + sp_axis_d2(1) + sp_axis_d2(2)   # nb04: 距離は加法分解
J = np.eye(Ms) - np.ones((Ms, Ms))/Ms
Bsp = -0.5 * J @ dsp2 @ J; Bsp = 0.5*(Bsp + Bsp.T)
evsp = np.sort(eigh(Bsp)[0])[::-1]
ratio = evsp/evsp[0]
print(f"空間 T³ の MDS 上位固有値比: {ratio[:9].round(3)}")
print(f"  有意固有値数(>5%): {int((ratio>0.05).sum())}")
print(f"  → 上位6個が等しい塊（T³ = 3軸 × 各 S¹ の2座標 = 埋め込み6次元）。")
print(f"    7個目で急落 → 内在次元は3（各 S¹ が内在1次元 ×3）。")
print()
print(f"時間: 被覆座標 t は1スカラー → timelike = 1（構成的に保証）。")
print(f"→ R×T³: 時間1（被覆）+ 空間3（T³ 内在）= (3+1)。")

空間 T³ の MDS 上位固有値比: [1.    1.    1.    1.    1.    1.    0.043 0.043 0.043]
  有意固有値数(>5%): 6
  → 上位6個が等しい塊（T³ = 3軸 × 各 S¹ の2座標 = 埋め込み6次元）。
    7個目で急落 → 内在次元は3（各 S¹ が内在1次元 ×3）。

時間: 被覆座標 t は1スカラー → timelike = 1（構成的に保証）。
→ R×T³: 時間1（被覆）+ 空間3（T³ 内在）= (3+1)。


## 16.3 内在次元の確定：T³ = 3 × S¹

空間 MDS の上位6固有値の塊が「3軸 × 各 S¹ の2座標」であることを、単一 S¹ の MDS と対照して
確定する。単一 S¹ は MDS で上位2座標（円）・内在1次元。3つの積 T³ は埋め込み6・内在3。

In [4]:
def circle_mds(m):
    th = np.linspace(0, 2*np.pi, m, endpoint=False)
    a = np.abs(np.cos(2*(th[:, None] - th[None, :])))/(2*N)
    with np.errstate(divide='ignore'):
        d = -np.log(a)
    d[~np.isfinite(d)] = 0.0; np.fill_diagonal(d, 0.0)
    J = np.eye(m) - np.ones((m, m))/m
    B = -0.5 * J @ (d**2) @ J
    return np.sort(eigh(0.5*(B + B.T))[0])[::-1]

ev1 = circle_mds(60)
print(f"単一 S¹ の MDS 上位固有値比: {(ev1[:4]/ev1[0]).round(3)}")
print(f"  → 上位2座標が支配（円）。内在1次元（残りは MDS リップル）。")
print(f"  T³ = S¹×S¹×S¹ → 埋め込み 2×3=6 座標、内在 1×3=3 次元。16.2 の6個塊と整合。")

単一 S¹ の MDS 上位固有値比: [1.    1.    0.358 0.358]
  → 上位2座標が支配（円）。内在1次元（残りは MDS リップル）。
  T³ = S¹×S¹×S¹ → 埋め込み 2×3=6 座標、内在 1×3=3 次元。16.2 の6個塊と整合。


## 16.4 因果順序：時間方向で大域的に推移的か

nb09 L-A では `sign(cos2θ)` の timelike 関係が非推移的（24% 破れ）で因果順序にならなかった。
R×T³ では時間＝被覆座標 t の大小（空間成分に依らない同時刻面の構造）であり、実数の全順序ゆえ
推移的になるはず。これを確認し、nb09 L-A の壁が解消されていることを示す。

In [5]:
nt = 8
t = np.linspace(0, 6*np.pi, nt)
order = t[:, None] < t[None, :]   # 時間順序（空間に依らない）
viol = tot = 0
for i in range(nt):
    for j in range(nt):
        for k in range(nt):
            if order[i, j] and order[j, k]:
                tot += 1; viol += (not order[i, k])
print(f"時間順序 t_i<t_j の推移性: {tot-viol}/{tot} "
      f"({100*(1-viol/max(tot,1)):.0f}%)")
print(f"  → 実数の全順序ゆえ完全推移。R×T³ で因果順序は時間方向に大域的に定義可。")
print(f"  nb09 L-A の非推移性（sign の周期反転による）は被覆時間で解消される。")

時間順序 t_i<t_j の推移性: 56/56 (100%)
  → 実数の全順序ゆえ完全推移。R×T³ で因果順序は時間方向に大域的に定義可。
  nb09 L-A の非推移性（sign の周期反転による）は被覆時間で解消される。


## 16.5 k* 構造の保持と非コンパクト性の局在

最後に、空間各軸が cos2θ の周期構造（k*=4、nb02）を保つこと、および非コンパクト性が時間方向に
**のみ**局在し空間 T³ はコンパクトであることを確認する。これは (3+1) 時空の自然な構造
（時間だけ非有界、空間はコンパクト）に一致する。

In [6]:
# k* 構造：空間軸の相関 cos2 のフーリエ主次数
m = 120
thh = np.linspace(0, 2*np.pi, m, endpoint=False)
row = np.cos(2*(thh - thh[0]))/(2*N)
F = np.abs(np.fft.rfft(row))
kmain = int(np.argmax(F[1:]) + 1)
print(f"空間軸の相関フーリエ主次数 = {kmain}（cos2θ の周期 2π/2 → nb02 の k* 構造）")
print()
print("非コンパクト性の局在：")
print("  時間 R: 被覆座標、全順序・非有界 → 非コンパクト")
print("  空間 T³: 各 S¹ は有界・周期 → コンパクト")
print("  → (3+1) の自然な構造（時間だけ非コンパクト、空間はコンパクトなトーラス）に一致。")

空間軸の相関フーリエ主次数 = 2（cos2θ の周期 2π/2 → nb02 の k* 構造）

非コンパクト性の局在：
  時間 R: 被覆座標、全順序・非有界 → 非コンパクト
  空間 T³: 各 S¹ は有界・周期 → コンパクト
  → (3+1) の自然な構造（時間だけ非コンパクト、空間はコンパクトなトーラス）に一致。


## 16.6 サマリー（established / open）

### established（この notebook で確定）

1. **R×T³ は (3+1) 時空として整合的に構成できる**（条件つき：独立設定本数 p=4 を認めた場合）。
   確認できた整合性：
   - **符号数 (3+1)**：時間1（被覆座標、構成的に1）＋ 空間3（T³ の MDS 上位6固有値の塊＝
     3軸×各 S¹ 2座標、内在3次元）。
   - **因果順序**：時間方向で全順序・100% 推移（nb09 L-A の非推移性を解消）。
   - **k* 構造**：空間各軸で cos2θ の周期構造を保持（nb02 と整合）。
   - **非コンパクト性の局在**：時間方向のみ非コンパクト、空間 T³ はコンパクト。

2. **すべて既確立結果と無矛盾**。nb13（被覆 II）・nb04（T^p の加法分解）・nb02（k*）・
   nb09（因果）の結果が R×T³ で矛盾なく組み合わさる。

3. **構成上の罠を明示**（16.1）。混合 squared interval の一括ローレンツ埋め込みは時間を
   多次元に割る人工物を生む。時間（被覆座標を直接与える）と空間（MDS で復元）を別々に
   構成するのが正しい（nb13 13.2 の手続き）。

### この成果の正確な射程

- **答えていること**：p=4（時空次元4）を認めれば、被覆 II の時間 R と T^3 の空間が
  (3+1) 時空として無矛盾に組める。次元原理（open-1）が目指す到達目標 R×T³ が具体的に固定された。
- **答えていないこと**：なぜ p=4 か（nb15 open-1、次元4の原理化）。これは依然未解決で、
  R×T³ の構成が整合的であることはこの問いを解かない。

### open（次へ）

1. **次元4の原理化（nb15 open-1、最重要・未解決）。** R×T³ が到達目標として固定された今、
   問いは「独立設定本数を 4 に強制する原理」に純化された。立場A（p=4 の根拠）か立場B
   （SO(3) の基礎づけ）を原理に格上げできるか。測定論的定式化が候補。

2. **R×T³ 上の作用と動力学。** 本丸の作用 `S=−βΣe^{−d}` を R×T³ 上で評価し、被覆時間方向の
   ダイナミクス（時間発展）が出るか。空間 T³ はコンパクトゆえ Kaluza–Klein 的なモード分解が
   可能か。**未着手。**

3. **本丸 no-go の解析的裏づけ（C-1）との接続。** R×T³ は「素の枠組みでは出ない」ものを
   追加原理（II＋次元）で組んだ構成。C-1 を固めるほど、この構成を追加原理として導入する
   正当性が増す（E-3）。

### 教訓（D への追記候補）

- **D-15（候補）：合成幾何は要素ごとに構成し、一括埋め込みを避ける。** 時間（被覆）と空間
  （MDS）は素性が違う。混ぜて一括の squared interval を埋め込むと人工物（時間の多次元化）が
  出る。各要素を正しい手続きで構成してから直積として合わせる。